Note: this notebook should be run with the r-seurat-h5 kernel

In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(rhdf5)
    library(arrow)
})

# Spot check a sample for completeness/consistency

In [47]:
sid <- "8073578341-02-02"

h5_file <- file.path("level3", paste0(sid, ".h5"))
pq_file <- file.path("level3", paste0(sid, ".parquet"))

stopifnot(file.exists(h5_file))
stopifnot(file.exists(pq_file))

# ---------------------------
# Read data back in
# ---------------------------
counts <- rhdf5::h5read(h5_file, "counts")
cells  <- rhdf5::h5read(h5_file, "cells")
genes  <- rhdf5::h5read(h5_file, "genes")
meta   <- arrow::read_parquet(pq_file)

# convert parquet result to plain data.frame if needed
meta <- as.data.frame(meta)

# attach dimnames to counts
stopifnot(length(dim(counts)) == 2)
stopifnot(nrow(counts) == length(genes))
stopifnot(ncol(counts) == length(cells))

rownames(counts) <- genes
colnames(counts) <- cells

# ---------------------------
# Basic completeness checks
# ---------------------------
cat("=== File existence ===\n")
cat("H5 exists:      ", file.exists(h5_file), "\n")
cat("Parquet exists: ", file.exists(pq_file), "\n\n")

cat("=== Dimensions ===\n")
cat("n_genes in H5:  ", nrow(counts), "\n")
cat("n_cells in H5:  ", ncol(counts), "\n")
cat("rows in meta:   ", nrow(meta), "\n\n")

cat("=== Required metadata columns ===\n")
cat("Has cell_id column: ", "cell_id" %in% colnames(meta), "\n")
cat("Has sid column:     ", "sid" %in% colnames(meta), "\n\n")

cat("=== Missingness / uniqueness ===\n")
cat("Any NA in genes:         ", any(is.na(genes)), "\n")
cat("Any NA in cells:         ", any(is.na(cells)), "\n")
cat("Duplicated genes:        ", anyDuplicated(genes), "\n")
cat("Duplicated cells:        ", anyDuplicated(cells), "\n")
cat("Duplicated meta cell_id: ",
    if ("cell_id" %in% colnames(meta)) anyDuplicated(meta$cell_id) else NA, "\n\n")

# ---------------------------
# Cross-check metadata vs H5
# ---------------------------
if (!("cell_id" %in% colnames(meta))) {
  stop("Metadata is missing required column: cell_id")
}

cells_in_h5_not_meta   <- setdiff(cells, meta$cell_id)
cells_in_meta_not_h5   <- setdiff(meta$cell_id, cells)
order_matches          <- all(cells == meta$cell_id)

cat("=== Cell ID agreement ===\n")
cat("Cells in H5 but not meta: ", length(cells_in_h5_not_meta), "\n")
cat("Cells in meta but not H5: ", length(cells_in_meta_not_h5), "\n")
cat("Cell order identical:     ", order_matches, "\n\n")

if (length(cells_in_h5_not_meta) > 0) {
  cat("First few cells in H5 not meta:\n")
  print(head(cells_in_h5_not_meta))
  cat("\n")
}

if (length(cells_in_meta_not_h5) > 0) {
  cat("First few cells in meta not H5:\n")
  print(head(cells_in_meta_not_h5))
  cat("\n")
}

# ---------------------------
# Check sid consistency
# ---------------------------
if ("sid" %in% colnames(meta)) {
  unique_sids <- unique(meta$sid)
  cat("=== sid consistency ===\n")
  cat("Unique sid values in parquet:\n")
  print(unique_sids)
  cat("All sid values match requested sid: ", all(meta$sid == sid), "\n\n")
}

# ---------------------------
# Check for empty / suspicious data
# ---------------------------
cat("=== Matrix content checks ===\n")
cat("Storage mode:            ", typeof(counts), "\n")
cat("Any NA in counts:        ", any(is.na(counts)), "\n")
cat("All-zero columns:        ", sum(colSums(counts) == 0), "\n")
cat("All-zero rows:           ", sum(rowSums(counts) == 0), "\n")
cat("Total counts:            ", sum(counts), "\n")
cat("Nonzero entries:         ", sum(counts != 0), "\n\n")

# ---------------------------
# Final pass/fail summary
# ---------------------------
ok <- TRUE

ok <- ok && nrow(counts) == length(genes)
ok <- ok && ncol(counts) == length(cells)
ok <- ok && !any(is.na(genes))
ok <- ok && !any(is.na(cells))
ok <- ok && anyDuplicated(genes) == 0
ok <- ok && anyDuplicated(cells) == 0
ok <- ok && anyDuplicated(meta$cell_id) == 0
ok <- ok && length(cells_in_h5_not_meta) == 0
ok <- ok && length(cells_in_meta_not_h5) == 0

if ("sid" %in% colnames(meta)) {
  ok <- ok && all(meta$sid == sid)
}

cat("=== Overall result ===\n")
if (ok) {
  cat("PASS: sample file appears complete and internally consistent.\n")
} else {
  cat("FAIL: sample file has one or more consistency problems.\n")
}

=== File existence ===
H5 exists:       TRUE 
Parquet exists:  TRUE 

=== Dimensions ===
n_genes in H5:   5100 
n_cells in H5:   118192 
rows in meta:    118192 

=== Required metadata columns ===
Has cell_id column:  TRUE 
Has sid column:      TRUE 

=== Missingness / uniqueness ===
Any NA in genes:          FALSE 
Any NA in cells:          FALSE 
Duplicated genes:         0 
Duplicated cells:         0 
Duplicated meta cell_id:  0 

=== Cell ID agreement ===
Cells in H5 but not meta:  0 
Cells in meta but not H5:  0 
Cell order identical:      TRUE 

=== sid consistency ===
Unique sid values in parquet:
[1] "8073578341-02-02"
All sid values match requested sid:  TRUE 

=== Matrix content checks ===
Storage mode:             double 
Any NA in counts:         FALSE 
All-zero columns:         0 
All-zero rows:            1 
Total counts:             6426127 
Nonzero entries:          5131274 

=== Overall result ===
PASS: sample file appears complete and internally consistent.
